<a href="https://colab.research.google.com/github/shreyaghora/ML/blob/main/Code/ML_Result01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install & Import  all the necessary libraries,functions, models


In [ ]:
!pip install xgboost
!pip install imblearn
!pip install openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.9/250.9 kB 9.3 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import make_scorer, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier


from imblearn.over_sampling import SMOTE, ADASYN

Loading the dataset from .csv to pandas's dataframe

In [ ]:
df = pd.read_csv('/content/bank-additional-full.csv', sep=';')

Preprocessing, Scaling, Sampling

In [ ]:
# Droping duration column to prevent leakage
df.drop('duration', axis=1, inplace=True) #drop method used for removing specified rows and column from a dataset


# For removing unknown values like 0, null, ?
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].replace('unknown', df[col].mode()[0])


#Encoding in binary[0 & 1] values i.e. Final value or, Target
df['y'] = df['y'].map({'no': 0, 'yes': 1})


# One-Hot Encoding: Convert ordinary data into numeric form [Only 1 column is (hot)1 at a time in each row for N categories]
df = pd.get_dummies(df, drop_first=True)


# Split Features(X:Input variables used for train the model and make prediction) & Target(y:Variable the model is intend to predict)

X = df.drop('y', axis=1)
y = df['y']


#Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


#Scaling: Step of preprocessing
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


#Sampling Techniques
smote = SMOTE(random_state=42)
adasyn = ADASYN(random_state=42)

X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)
X_train_ad, y_train_ad = adasyn.fit_resample(X_train, y_train)


#Models
models = {

    "Decision Tree": DecisionTreeClassifier
     (
            max_depth=8,
            min_samples_split=10,
            min_samples_leaf=4,
            random_state=42

    ),
    "Random Forest": RandomForestClassifier(random_state=42, n_estimators=300),

   "MLP": MLPClassifier(
       max_iter=500, random_state=42, early_stopping=True,hidden_layer_sizes=(128,64)
                  ),
    "XGBoost": XGBClassifier( eval_metric='logloss', random_state=42),


}

# Evaluation Metrics
scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'roc_auc': 'roc_auc'
}


#Cross Validation Setup
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
#For storing
results= []
#Function to Evaluate Models
def evaluate_models(X, y, label):
    print(f"\n===== {label} =====")
    for name, model in models.items():
        scores = cross_validate(model, X, y, cv=skf, scoring=scoring)

        metrics={
            "Label":f"{label}",
            "Model": name,
            "Accuracy":  f"{np.mean(scores['test_accuracy']):.4f}",
            "Precision": f"{np.mean(scores['test_precision']):.4f}",
            "Recall":    f"{np.mean(scores['test_recall']):.4f}",
            "F1 Score":  f"{np.mean(scores['test_f1']):.4f}",
            "ROC-AUC":   f"{np.mean(scores['test_roc_auc']):.4f}"
        }
        results.append(metrics)

        print(f"\nModel: {name}")
        print(f"Accuracy:  {np.mean(scores['test_accuracy']):.4f}")
        print(f"Precision: {np.mean(scores['test_precision']):.4f}")
        print(f"Recall:    {np.mean(scores['test_recall']):.4f}")
        print(f"F1 Score:  {np.mean(scores['test_f1']):.4f}")
        print(f"ROC-AUC:   {np.mean(scores['test_roc_auc']):.4f}")

#Run Experiments

# Baseline(Without sampling)
evaluate_models(X_train, y_train, "Baseline (No Sampling)")


#(With Sampling)

# Result of using SMOTE algo.
evaluate_models(X_train_sm, y_train_sm, "SMOTE")

# Result of using ADASYN algo.
evaluate_models(X_train_ad, y_train_ad, "ADASYN")
#Converting into dataframe
results_df = pd.DataFrame(results)
from google.colab import drive
# Save directly to Drive
results_df.to_excel('/content/drive/MyDrive/mlresult01.xlsx', index=False)


===== Baseline (No Sampling) =====

Model: Decision Tree
Accuracy:  0.8987
Precision: 0.6142
Recall:    0.2713
F1 Score:  0.3758
ROC-AUC:   0.7731

Model: Random Forest
Accuracy:  0.8918
Precision: 0.5391
Recall:    0.2810
F1 Score:  0.3690
ROC-AUC:   0.7724

Model: MLP
Accuracy:  0.8973
Precision: 0.6228
Recall:    0.2306
F1 Score:  0.3348
ROC-AUC:   0.7833

Model: XGBoost
Accuracy:  0.8964
Precision: 0.5879
Recall:    0.2724
F1 Score:  0.3717
ROC-AUC:   0.7863

===== SMOTE =====

Model: Decision Tree
Accuracy:  0.8513
Precision: 0.9113
Recall:    0.7784
F1 Score:  0.8396
ROC-AUC:   0.9052

Model: Random Forest
Accuracy:  0.9406
Precision: 0.9486
Recall:    0.9317
F1 Score:  0.9400
ROC-AUC:   0.9826

Model: MLP
Accuracy:  0.9076
Precision: 0.8968
Recall:    0.9216
F1 Score:  0.9089
ROC-AUC:   0.9636

Model: XGBoost
Accuracy:  0.9380
Precision: 0.9683
Recall:    0.9057
F1 Score:  0.9359
ROC-AUC:   0.9739

===== ADASYN =====

Model: Decision Tree
Accuracy:  0.8484
Precision: 0.9105
Rec